В данном домашнем задании вам необходимо:

1. Используйте датасет "Собаки и кошки", рассмотренный в данном уроке. Причем используйте его целиком, а не только 4000 изображений.
2. Проведите аугментацию изображений.
3. В качестве предобученной модели возьмите `MobileNet`
4. Создайте модель, приведенную ниже.
5. Обучите модель и проверьте на тестовой выборке.
6. Если модель не обеспечивает заданную точность - "поиграйтесь" с гиперпараметрами.


Для получения 3 баллов за задание необходимо достичь на контрольной выборке точности 90%, 4 баллов -  более 93%, 5 баллов - более 95%.

На 20 тыс. изображений данная модель выдавала нам результат 99%.

**Подсказка**. Обратите внимание, что предлагаемая модель уже не является бинарной классификацией. Это уже задача многоклассовой классификации (в нашем случае 2 класса). А значит в генераторах изображений необходимо использовать:

```pyton
class_mode='categorical'
```

Также необходимо вспомнить какую функцию ошибки использовать с задачей многоклассовой классификации. Можно попробовать в качестве оптимизатора использовать Adam с разными шагами.

Также обратите внимание, что вместо слоя `Flatten()`, вам предлагается использовать `GlobalAveragePooling2D()` (https://keras.io/api/layers/pooling_layers/global_average_pooling2d/).


In [ ]:
from keras.applications import MobileNet
from keras import models
from keras.layers import GlobalAveragePooling2D, Dense, Dropout
from keras import optimizers

def model_maker():
    base_model = MobileNet(include_top=False, input_shape = (IMG_WIDTH, IMG_HEIGHT, 3))

    for layer in base_model.layers[:]:
        layer.trainable = False

    input = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    custom_model = base_model(input)
    custom_model = GlobalAveragePooling2D()(custom_model)
    custom_model = Dense(64, activation='relu')(custom_model)
    custom_model = Dropout(0.5)(custom_model)
    predictions = Dense(NUM_CLASSES, activation='softmax')(custom_model)

    return Model(inputs=input, outputs=predictions)

In [ ]:
model_maker.summary()

In [6]:
# Импорт

!pip install -q kaggle

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import shutil
import random
from sklearn.model_selection import train_test_split

# Импорты для Keras 3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Для Google Colab
from google.colab import files

print("Библиотеки импортированы!")

# Загрузка

# Проверяем, есть ли уже файлы с данными
if not os.path.exists('/content/train.zip'):
    print("Скачиваем датасет с Kaggle...")

    # Пытаемся скачать
    !kaggle competitions download -c dogs-vs-cats

    # Проверяем, скачался ли файл
    if not os.path.exists('dogs-vs-cats.zip'):
        print("Не удалось скачать с Kaggle.")
        print("Загрузите файлы вручную:")
        print("1. Скачайте train.zip с https://www.kaggle.com/competitions/dogs-vs-cats/data")
        print("2. Загрузите его через files.upload()")

        uploaded = files.upload()
        # Переименовываем загруженный файл
        for filename in uploaded.keys():
            if filename.endswith('.zip'):
                os.rename(filename, '/content/train.zip')
                break
else:
    print("Датасет уже есть в /content/train.zip")

# Распаковка
if os.path.exists('dogs-vs-cats.zip'):
    with zipfile.ZipFile('dogs-vs-cats.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')

if os.path.exists('/content/train.zip'):
    with zipfile.ZipFile('/content/train.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Датасет распакован!")
else:
    print("Файл train.zip не найден!")
    print("Пожалуйста, загрузите его вручную.")

# Проверка

if not os.path.exists('/content/train'):
    print("⚠️ Папка /content/train не найдена!")
    print("Создаем тестовый датасет из доступных файлов...")

    # Ищем любые изображения в папке
    test_source = '/content'
    for root, dirs, files in os.walk(test_source):
        for file in files:
            if file.endswith('.jpg') or file.endswith('.png'):
                print(f"Найден файл: {file}")

    # Если ничего не найдено, предлагаем загрузить вручную
    print("\nПожалуйста, загрузите train.zip вручную через files.upload()")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/train')
            break

# Структура

BASE_DIR = '/content/dogs_vs_cats'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VALIDATION_DIR = os.path.join(BASE_DIR, 'validation')
TEST_DIR = os.path.join(BASE_DIR, 'test')

# Очищаем старые папки
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

# Создаем структуру
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

for cls in ['cats', 'dogs']:
    os.makedirs(os.path.join(TRAIN_DIR, cls), exist_ok=True)
    os.makedirs(os.path.join(VALIDATION_DIR, cls), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, cls), exist_ok=True)

print("Структура папок создана!")

# Данные

source_dir = '/content/train'
if os.path.exists(source_dir):
    train_files = os.listdir(source_dir)
    train_files = [f for f in train_files if f.endswith('.jpg')]
else:
    train_files = []
    print(f"Папка {source_dir} не найдена!")

if len(train_files) == 0:
    print("Изображения не найдены!")
    print("Создаем структуру с минимальным количеством файлов для теста...")

    # Создаем несколько тестовых файлов, если их нет
    for i in range(10):
        with open(f'/content/train/cat.{i}.jpg', 'w') as f:
            f.write('dummy content')
        with open(f'/content/train/dog.{i}.jpg', 'w') as f:
            f.write('dummy content')

    train_files = os.listdir(source_dir)
    train_files = [f for f in train_files if f.endswith('.jpg')]

print(f"Всего изображений: {len(train_files)}")

# Разбиваем на классы
cat_files = [f for f in train_files if f.startswith('cat')]
dog_files = [f for f in train_files if f.startswith('dog')]

print(f"Котов: {len(cat_files)}")
print(f"Собак: {len(dog_files)}")

# Функция для копирования файлов
def copy_files(file_list, source_dir, dest_dir):
    for f in file_list:
        src = os.path.join(source_dir, f)
        dst = os.path.join(dest_dir, f)
        if os.path.exists(src):
            shutil.copy(src, dst)

# Разбиваем данные
def split_data(files, train_ratio=0.7, val_ratio=0.15):
    train, temp = train_test_split(files, test_size=(1 - train_ratio), random_state=42)
    val, test = train_test_split(temp, test_size=(val_ratio / (val_ratio + (1 - train_ratio - val_ratio))), random_state=42)
    return train, val, test

if len(cat_files) > 0:
    cat_train, cat_val, cat_test = split_data(cat_files)
    dog_train, dog_val, dog_test = split_data(dog_files)

    # Копируем файлы
    copy_files(cat_train, source_dir, os.path.join(TRAIN_DIR, 'cats'))
    copy_files(cat_val, source_dir, os.path.join(VALIDATION_DIR, 'cats'))
    copy_files(cat_test, source_dir, os.path.join(TEST_DIR, 'cats'))

    copy_files(dog_train, source_dir, os.path.join(TRAIN_DIR, 'dogs'))
    copy_files(dog_val, source_dir, os.path.join(VALIDATION_DIR, 'dogs'))
    copy_files(dog_test, source_dir, os.path.join(TEST_DIR, 'dogs'))

    print(f"Кошки: Train={len(cat_train)}, Val={len(cat_val)}, Test={len(cat_test)}")
    print(f"Собаки: Train={len(dog_train)}, Val={len(dog_val)}, Test={len(dog_test)}")
else:
    print("Не удалось разбить данные: нет файлов!")

print("Данные распределены по папкам!")

# Параметры

IMG_WIDTH = 224
IMG_HEIGHT = 224
NUM_CLASSES = 2
BATCH_SIZE = 32
EPOCHS = 30

print(f"Размер изображений: {IMG_WIDTH}x{IMG_HEIGHT}")
print(f"Количество классов: {NUM_CLASSES}")

# Аргументы

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_test_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Train samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

# Создание и компляция

def create_model():
    base_model = MobileNet(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_WIDTH, IMG_HEIGHT, 3)
    )

    for layer in base_model.layers:
        layer.trainable = False

    input_tensor = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    x = base_model(input_tensor)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inputs=input_tensor, outputs=outputs)

model = create_model()
model.summary()

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("Модель создана и скомпилирована!")

# Колбэки

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Обучение

print("\nНачинаем обучение...")
history = model.fit(
    train_generator,
    steps_per_epoch=max(1, train_generator.samples // BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=max(1, validation_generator.samples // BATCH_SIZE),
    callbacks=callbacks,
    verbose=1
)

print("Обучение завершено!")

# Настройка

print("\n🔧 Начинаем fine-tuning...")
model.load_weights('best_model.h5')

base_model = model.layers[1]
for layer in base_model.layers[:-10]:
    layer.trainable = False
for layer in base_model.layers[-10:]:
    layer.trainable = True

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=1e-5),
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    steps_per_epoch=max(1, train_generator.samples // BATCH_SIZE),
    epochs=15,
    validation_data=validation_generator,
    validation_steps=max(1, validation_generator.samples // BATCH_SIZE),
    callbacks=callbacks,
    verbose=1
)

model.load_weights('best_model.h5')
print("Fine-tuning завершен!")

# Оценка

test_loss, test_accuracy = model.evaluate(
    test_generator,
    steps=max(1, test_generator.samples // BATCH_SIZE),
    verbose=1
)

print(f"\n{'='*50}")
print(f"📊 ТЕСТОВАЯ ТОЧНОСТЬ: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"📉 ТЕСТОВЫЕ ПОТЕРИ: {test_loss:.4f}")
print(f"{'='*50}")

if test_accuracy >= 0.95:
    print("5 БАЛЛОВ! Отлично!")
elif test_accuracy >= 0.93:
    print("4 БАЛЛА! Хорошо!")
elif test_accuracy >= 0.90:
    print("3 БАЛЛА! Неплохо!")
else:
    print("Попробуйте увеличить эпохи или изменить архитектуру.")

# Визуал

def plot_history(history, title="Training History"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{title} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{title} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history, "Initial Training")
plot_history(history_fine, "Fine-Tuning")

# Сохранение

model.save('final_model.h5')
print("\nМодель сохранена как 'final_model.h5'")
print(f"Итоговая точность: {test_accuracy*100:.2f}%")

try:
    files.download('final_model.h5')
except:
    print("Не удалось скачать модель автоматически.")
    print("Модель сохранена в папке /content/")

Библиотеки импортированы!
Скачиваем датасет с Kaggle...
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
⚠️ Не удалось скачать с Kaggle.
Загрузите файлы вручную:
1. Скачайте train.zip с https://www.kaggle.com/competitions/dogs-vs-cats/data
2. Загрузите его через files.upload()


KeyboardInterrupt: 